# Grid trading (S-080) — viewer (2026-09-07)

Pre-registered in [README.md](README.md); verdict in [findings.md](findings.md). The four cells (0.2% / 0.5% spacing × maker / taker costs) were run once on `btc_1m` 2020-01-01 → 2026-09-06 (`results/run_1m.log`). Re-running takes ≈ 25 min on 3.5M bars: `python grid_harness.py --bars 1m`; the 5m variant is a quick check.

## Summary table (as produced)

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
s = pd.read_csv('results/grid_summary.csv').set_index('cell')
display(s.round(2))

## Equity curves

Cumulative % of capital (additive, unlevered), per cell.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 5))
for cell in s.index:
    d = pd.read_csv(f'results/grid_daily_{cell}.csv', index_col=0, parse_dates=True)
    ax.plot(d.index, d['equity'] * 100, label=cell)
ax.axhline(0, color='k', lw=0.5); ax.legend(); ax.set_ylabel('% of capital')
ax.set_title('Grid P&L, realized + mark-to-market of open lots')
plt.show()

## Per-year returns

The only maker cell with positive Sharpe in both halves (A_maker_0.5) is positive in every year except 2022, where the long inventory accumulated into the decline cost −39% and sets the 62% max drawdown.

In [ ]:
rows = {}
for cell in s.index:
    d = pd.read_csv(f'results/grid_daily_{cell}.csv', index_col=0, parse_dates=True)
    rows[cell] = d['ret'].groupby(d.index.year).sum() * 100
display(pd.DataFrame(rows).round(1))

## Re-run (optional)

Uncomment to recompute everything read-only against `prod.db`.

In [ ]:
# %run grid_harness.py --bars 5m